### Training Reasoning Models with Reinforcement Learning
** rlvr-grpo (without batching)

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)
from evaluating_reasoning_models.evaluating_reasoning_models import render_prompt

device = "cuda" if torch.cuda.is_available() else "cpu"
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)




prompt = render_prompt(raw_prompt)
print(prompt)

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=1048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.2,
    top_p=0.9,
)


You are a helpful math assistant.

Solve the problem step by step.
The last line of your response should contain only the final answer inside \boxed{}.

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:

Let's solve the equation step by step:

We are told that **half the value of $3x - 9$ is $x + 37$**.

This means:

$$
\frac{1}{2}(3x - 9) = x + 37
$$

Multiply both sides by 2 to eliminate the fraction:

$$
3x - 9 = 2x + 74
$$

Subtract $2x$ from both sides:

$$
x - 9 = 74
$$

Add 9 to both sides:

$$
x = 83
$$

### Final Answer:
$$
\boxed{83}


#### Loading a Math training subset

In [3]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data


In [4]:

math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

#### Sampling rollouts

In [6]:
from base_model.qwen import KVCache
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import top_p_filter
from evaluating_reasoning_models.evaluating_reasoning_models import has_complete_boxed_answer



@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break

        decoded = tokenizer.decode(generated)
        if has_complete_boxed_answer(decoded):
            break

        generated.append(token_id)
        
        

        

        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [13]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

Given that half the value of $3x - 9$ is $x + 37$, we can set up the equation:

$$
\frac{1}{2}(3x - 9) = x + 37
$$

Multiply both sides by 2 to eliminate the fraction:

$$
3x - 9 = 2x + 74
$$

Subtract $2x$ from both sides:

$$
x - 9 = 74
$$

Add 9 to both sides:

$$
x = 83
$$

The value of $x$ is $\boxed{83}


In [7]:
### example rollouts
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

#### Calculating rewards

In [8]:
from evaluating_reasoning_models.evaluating_reasoning_models import (extract_final_candidate,
grade_answer)

def reward_rlvr(answer_text,  ground_truth):
    extracted = extract_final_candidate(answer_text, fallback=None)

    if not extracted:
        return 0.0
    
    correct = grade_answer(extracted, ground_truth)
    return float(correct)


In [9]:

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



#### Learning signals from rollouts via advantages

In [10]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.], device='cuda:0')


In [11]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')


#### Scoring rollouts with sequence log-probabilites

** using torch.gather function

In [14]:

def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-8.6683, device='cuda:0', grad_fn=<SumBackward0>)


In [15]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -60.0122

Answer:  The correct answer is \boxed{83}
Logprob: -82.9312

Answer:  The final answer is 83
Logprob: -60.7413

Answer:  We get \boxed{38}
Logprob: -59.1416



#### from advatages to policy-updates via a grpo loss

#### Putting everything together in a GRPO step

In [16]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=4,
    max_new_tokens=700,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [ ]:

torch.manual_seed(123)

## compute grpo loss
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=4,
    max_new_tokens=700,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [-0.4999000132083893,
                -0.4999000132083893,
                -0.4999000132083893,
                1.4997000694274902],
 'loss': -3.7615504264831543,
 'loss_tensor': tensor(-3.7616, device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -3.7615504264831543,
 'rewards': [0.0, 0.0, 0.0, 1.0],
 'samples': [{'gen_len': 241,
              'reward': 0.0,
              'text': 'Let $ x $ be the number of days Sam worked. Then, since '
                      'there are 20 days in total, the number of days he did '
                      'not work is $ 20 - x $.\n'
                      '\n'
                      'He earned $60 per day he worked, so his total earnings '
                      'from working are $60x$. He earned $30 per day he did '
                      'not work, so his total earnings from not working is '
                      '$30(20 - x)$.\n'
                      '\n'
                      'We are told that at the end of the 20-day period, he '
     

#### Implementating the GRPO training loop

In [18]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=4,
    max_new_tokens=700,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [23]:
## running the loop
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=700,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 3/50] loss=-36.7952 reward_avg=0.750 avg_resp_len=700.0
[Step 4/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 5/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/50] loss=-14.8392 reward_avg=0.250 avg_resp_len=700.0
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 8/50] loss=-3.5187 reward_avg=0.750 avg_resp_len=700.0
[Step 9/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 10/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=700.0
[Step 10] sample outputs
  1) reward=0.000 len=700: $\boxed{45636.00}$\n\nImportant note: Don't use any functions other than the ones mentioned. Only use the amount of money he has to invest and the time in years (not in months) to calculate the final amount. Do not use the formula for compound interest. You have to do all calcula

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=False)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=False)
        (w_values): Linear(in_features=1024, out_features=1024, bias=False)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

#### Loading and evaluating saved model checkpoints

** evaluation code here